# Multi-Platform Cosmetics Market Analysis | Myntra Data Cleaning

Author: Neethu Raj

## Project Overview

This project analyzes the Indian online cosmetics market using product data collected from multiple e-commerce platforms.

The Nykaa product dataset was previously scraped, cleaned, explored, and analyzed separately. Myntra data was subsequently collected to extend the analysis across platforms.

This notebook focuses on cleaning and validating the scraped Myntra product dataset. The cleaned dataset will be used for cross-platform analysis with the cleaned Nykaa dataset.

## Project Progress

| Phase | Status |
|---|:---:|
| 1. Web Scraping – Nykaa | ✅ Completed |
| 2. Data Cleaning – Nykaa | ✅ Completed |
| 3. Exploratory Data Analysis – Nykaa | ✅ Completed |
| 4. SQL Business Analysis – Nykaa | ✅ Completed |
| 5. Power BI Dashboard – Nykaa | ✅ Completed |
| 6. Web Scraping – Myntra | ✅ Completed |
| 7. Data Cleaning – Myntra | 🔄 In Progress |
| 8. Cross-Platform Price & Discount Comparison | ⏳ Planned |
| 9. Cross-Platform Power BI Dashboard | ⏳ Planned |

## Dataset Information

- **Source:** Self-scraped product data from Myntra using Python and Selenium.
- **Total records before cleaning:** 80,740
- **Number of features before cleaning:** 11

The scraped dataset contains product-level information such as:

- Product Name
- Product URL
- Category
- Sub-category
- Brand
- Original Price
- Discounted Price
- Discount
- Rating
- Rating Count
- Platform

Each row represents one product listing collected from Myntra.

The cleaned dataset will be used for cross-platform analysis with the previously cleaned Nykaa dataset.

## Objectives

The objectives of this notebook are to:

- Inspect the structure and quality of the scraped Myntra dataset.
- Convert relevant columns to appropriate data types.
- Handle missing values appropriately.
- Identify and assess duplicate records and URLs.
- Examine duplicate product names and determine whether they represent legitimate listings.
- Validate pricing and rating-related fields using logical checks.
- Prepare a clean and consistent Myntra dataset for downstream analysis.

## Contents

1. Import Libraries
2. Load Dataset
3. Initial Data Exploration
   - 3.1 Data Structure
   - 3.2 Dataset Dimensions
   - 3.3 Data Types
4. Data Cleaning
   - 4.1 Change Data Types
   - 4.2 Handle Missing Values
   - 4.3 Deal with Duplicates
5. Logical Validation
6. Export Cleaned Dataset
7. Conclusion

## 1. Import Libraries

The required Python libraries are imported for data manipulation, data cleaning, and validation of the scraped Myntra dataset.

In [1]:
import pandas as pd

## 2. Load Dataset

The scraped Myntra product dataset is loaded into Pandas for preprocessing and validation.

The dataset contains product listings collected from Myntra using the web-scraping process completed prior to this notebook.

In [2]:
myntra_df = pd.read_csv("myntra_listing.csv")

## 3. Initial Data Exploration

The initial structure and quality of the scraped Myntra dataset are examined before applying cleaning operations.

This includes checking the dataset dimensions, data types, and missing values to identify fields that require preprocessing.

### 3.1. Data Structure

The first few records and the overall structure of the dataset are inspected to understand the available columns and the type of information collected for each product listing.

In [3]:
myntra_df.head()

,product_name,url,category,sub_category,brand,discounted_price,original_price,discount_percent,rating_out_of_5,rating_count,platform
0,Flawless Fusion Bronzer-05,https://www.myntra.com/bronzer/daily+life+fore...,Face Makeup,Bronzer,Daily Life Forever52,Rs. 390,Rs. 549,29.0,4.7,|149,Myntra
1,Mega Bronzer - Warm 02,https://www.myntra.com/bronzer/makeup+revoluti...,Face Makeup,Bronzer,Makeup Revolution London,Rs. 693,Rs. 795,12.8,4.5,|1k,Myntra
2,Must Have Face Palette - 12g,https://www.myntra.com/bronzer/iba/iba-must-ha...,Face Makeup,Bronzer,Iba,Rs. 764,Rs. 925,17.4,4.0,|16,Myntra
3,Bronzer - BNZ001 - 18.8 g,https://www.myntra.com/bronzer/character/chara...,Face Makeup,Bronzer,Character,Rs. 679,Rs. 849,20.0,NaN,NaN,Myntra
4,Bronzer - BNZ003 - 18.8 g,https://www.myntra.com/bronzer/character/chara...,Face Makeup,Bronzer,Character,Rs. 679,Rs. 849,20.0,NaN,NaN,Myntra


### 3.2. Checking Dataset Dimensions

The number of rows and columns is examined to understand the size of the scraped Myntra dataset and the number of attributes available for each product listing.

In [4]:
myntra_df.shape

(80740, 11)

#### Observation

The dataset contains **80,740 product listings and 11 columns**.

Myntra typically represents different product variants, such as shades or sizes, as separate listings. Therefore, product counts in this dataset represent **product listings rather than unique product families**.

This distinction is important when the Myntra dataset is later compared with Nykaa, where multiple shades may be represented within a single product listing.

### 3.3. Checking Data Types

The data types of all columns are examined to identify fields that require conversion before further processing.

Correct data types are necessary for accurate numerical calculations, filtering, aggregation, and validation.

In [5]:
myntra_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 80740 entries, 0 to 80739
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   product_name      80740 non-null  str    
 1   url               80740 non-null  str    
 2   category          80740 non-null  str    
 3   sub_category      80740 non-null  str    
 4   brand             80740 non-null  str    
 5   discounted_price  80740 non-null  str    
 6   original_price    80740 non-null  str    
 7   discount_percent  80740 non-null  float64
 8   rating_out_of_5   36807 non-null  float64
 9   rating_count      36807 non-null  str    
 10  platform          80740 non-null  str    
dtypes: float64(2), str(9)
memory usage: 6.8 MB


#### Observation

The price and rating count columns contain values that require conversion to numeric data types.

The discount field is not used directly because its scraped values are not consistently represented. The discount percentage is therefore derived from the original and discounted prices during the cleaning process.

Text-based fields such as product names, categories, subcategories, brands, and URLs are retained as text.

## 4. Data Cleaning

The scraped Myntra dataset is cleaned to improve data consistency and prepare it for downstream analysis.

The cleaning process includes converting numerical fields to appropriate data types, examining missing values, and checking for duplicate records and URLs.

### 4.1. Change Data Types

Price and rating count columns are converted to appropriate numeric data types.

Non-numeric characters and formatting symbols present in the scraped values are removed before conversion.

#### 4.1.1. Convert `discounted_price` to a Numeric Type

The `discounted_price` column is cleaned by removing the currency prefix and converting the resulting values to a numeric data type.

In [6]:
myntra_df['discounted_price'].dtype

<StringDtype(storage='python', na_value=nan)>

In [7]:
myntra_df['discounted_price'][0]

'Rs. 390'

In [8]:
myntra_df['discounted_price'] = (
    myntra_df['discounted_price']
    .str.replace('Rs.', '', regex=False)
    .str.strip()
    .astype(float) 
)

#### 4.1.2. Convert `original_price` to a Numeric Type

The `original_price` column is cleaned by removing the currency prefix and converting the resulting values to a numeric data type.

In [9]:
myntra_df['original_price'].dtype

<StringDtype(storage='python', na_value=nan)>

In [10]:
myntra_df['original_price'][0]

'Rs. 549'

In [11]:
myntra_df['original_price'] = (
    myntra_df['original_price']
    .str.replace('Rs.', '', regex=False)
    .str.strip()
    .astype(float) 
)

#### 4.1.3. Convert `rating_count` to a Numeric Type

The `rating_count` column is converted to a numeric data type.

Scraped values containing formatting characters and abbreviated values such as `k` are converted into their corresponding numeric values to support further analysis.

In [12]:
myntra_df['rating_count'].dtype


<StringDtype(storage='python', na_value=nan)>

In [13]:
myntra_df['rating_count'][0]

'|149'

In [14]:
myntra_df['rating_count'] = (
    myntra_df['rating_count']
    .str.replace('|', '', regex=False)
    .str.strip()
    .apply(
        lambda x: int(float(x[:-1]) * 1000)
        if pd.notna(x) and x.endswith('k')
        else int(float(x))
        if pd.notna(x)
        else pd.NA
    )
    .astype('Int64')
)

In [15]:
myntra_df.describe()

,discounted_price,original_price,discount_percent,rating_out_of_5,rating_count
count,80740.000000,80740.000000,80740.000000,36807.000000,36807.0
mean,532.735136,738.679736,27.492156,4.255815,3397.421523
std,812.827582,863.264435,23.813623,0.426131,19174.678072
min,43.000000,49.000000,0.000000,1.000000,5.0
25%,244.000000,379.000000,3.800000,4.100000,19.0
50%,340.000000,499.000000,24.700000,4.400000,76.0
75%,510.000000,799.000000,45.900000,4.500000,499.0
max,42150.000000,42150.000000,89.900000,5.000000,328300.0


### 4.2. Dealing with Missing Values

Missing values are examined to understand which product attributes are unavailable in the scraped Myntra dataset.

Missing values are not automatically replaced because their absence may represent information that was not displayed or available for a particular product listing.

In [16]:
myntra_df.isnull().sum()

product_name            0
url                     0
category                0
sub_category            0
brand                   0
discounted_price        0
original_price          0
discount_percent        0
rating_out_of_5     43933
rating_count        43933
platform                0
dtype: int64

#### Observation

Out of **80,740 product listings, 43,933 have missing rating-related information**.

For these listings, the rating and/or rating count information was not available in the scraped data, which may indicate that the information was not displayed or available on the product listing at the time of collection.

#### 4.2.1. Check Missing Ratings and Rating Count

The missing values in `rating_out_of_5` and `rating_count` are examined together to understand the availability of customer rating information across product listings.

In [17]:
myntra_df[
    ["rating_out_of_5","rating_count"]
].isna().value_counts()

rating_out_of_5  rating_count
True             True            43933
False            False           36807
Name: count, dtype: int64

### 4.3. Deal with Duplicates

Duplicate records are examined to determine whether the scraped dataset contains repeated product listings.

Both complete duplicate records and duplicate URLs are checked because identical product names do not necessarily indicate duplicate products on Myntra.

#### 4.3.1. Checking Duplicate Records

Complete duplicate records are identified by comparing all columns in the dataset.

This check helps determine whether the scraping process produced identical rows more than once.

In [18]:
pd.set_option('display.max_colwidth', None)

In [19]:
myntra_df[myntra_df.duplicated()]

,product_name,url,category,sub_category,brand,discounted_price,original_price,discount_percent,rating_out_of_5,rating_count,platform


#### Observation

No complete duplicate records were found in the dataset.

This indicates that there are no rows with identical values across all available columns.

#### 4.3.2. Checking for Duplicate URLs

Product URLs are checked separately because the same product name may appear across different listings, while the URL provides a more reliable identifier for an individual product listing.

In [20]:
myntra_df[myntra_df['url'].duplicated()]

,product_name,url,category,sub_category,brand,discounted_price,original_price,discount_percent,rating_out_of_5,rating_count,platform


In [21]:
myntra_df["url"].nunique()

80740

#### Observation

No duplicate URLs were found in the dataset.

All **80,740 product listings have unique URLs**, indicating that each scraped listing corresponds to a distinct product page URL.

#### 4.3.3. Duplicate Product Names

Product names are examined separately from URLs to determine whether repeated product names represent duplicate records or legitimate separate listings.

In [22]:
myntra_df[myntra_df['product_name'].duplicated()]

,product_name,url,category,sub_category,brand,discounted_price,original_price,discount_percent,rating_out_of_5,rating_count,platform
12,Full Heat Matte Bronzer,https://www.myntra.com/bronzer/bh+cosmetics/bh-cosmetics-los-angeles-full-heat-matte-bronzer---tan-tuscany/24843440/buy,Face Makeup,Bronzer,BH COSMETICS,595.0,595.0,0.0,4.6,31,Myntra
14,Full Heat Matte Bronzer,https://www.myntra.com/bronzer/bh+cosmetics/bh-cosmetics-los-angeles-full-heat-matte-bronzer---caramel-cabo/24843372/buy,Face Makeup,Bronzer,BH COSMETICS,595.0,595.0,0.0,4.6,31,Myntra
28,Full Heat Matte Bronzer,https://www.myntra.com/bronzer/bh+cosmetics/bh-cosmetics-los-angeles-full-heat-matte-bronzer---honey-heights/24843442/buy,Face Makeup,Bronzer,BH COSMETICS,595.0,595.0,0.0,4.6,31,Myntra
39,Long Lasting Baked Bronzer -7g,https://www.myntra.com/bronzer/milani/milani-long-lasting-baked-bronzer---7-g---capri-copper---03/41804347/buy,Face Makeup,Bronzer,MILANI,1506.0,1950.0,22.8,4.3,40,Myntra
44,Long Lasting Baked Bronzer -7g,https://www.myntra.com/bronzer/milani/milani-long-lasting-baked-bronzer---7-g---sicilian-sunset---04/41804344/buy,Face Makeup,Bronzer,MILANI,1506.0,1950.0,22.8,4.3,40,Myntra
...,...,...,...,...,...,...,...,...,...,...,...
80716,Marvel Gel Nail Lacquer,https://www.myntra.com/nail-polish/kafi/kafi-shine-pro-long-lasting-marvel-gel-nail-lacquer---nights-of-northen-lights/17623926/buy,Nail Makeup,Nail Polish,KAFI,159.0,295.0,46.1,4.3,75,Myntra
80721,Shine PRO Nail Lacquer,https://www.myntra.com/nail-polish/kafi/kafi-shine-pro-long-lasting-glitter-nail-lacquer--i-am-bulletproof-10ml/17605028/buy,Nail Makeup,Nail Polish,KAFI,151.0,225.0,32.9,4.1,110,Myntra
80734,Colour Nail Polish,https://www.myntra.com/nail-polish/cuccio/cuccio-colour-nail-polish---positively-positano-/17587626/buy,Nail Makeup,Nail Polish,Cuccio,384.0,749.0,48.7,4.1,1100,Myntra
80735,Colour Nail Paint,https://www.myntra.com/nail-polish/cuccio/cuccio-colour-nail-polish---touch-of-evil/17587620/buy,Nail Makeup,Nail Polish,Cuccio,399.0,749.0,46.7,4.1,1100,Myntra


In [23]:
myntra_df[myntra_df['product_name'].duplicated()]["product_name"].count()

np.int64(30688)

In [24]:
myntra_df[myntra_df['product_name'].duplicated(keep=False)] \
    .sort_values('product_name') \
    [['product_name', 'brand', 'url']].head(20)

,product_name,brand,url
56954,#NAME?,Pixi,https://www.myntra.com/lip-balm/pixi/pixi-hydra-lip-treat-tinted-lip-balm-with-hyaluronic-acid--shea-butter---peachy/37205350/buy
56799,#NAME?,Pixi,https://www.myntra.com/lip-balm/pixi/pixi-hydra-lip-treat-tinted-lip-balm-with-hyaluronic-acid--shea-butter---poppy/37205353/buy
56927,#NAME?,Pixi,https://www.myntra.com/lip-balm/pixi/pixi-hydra-lip-treat-tinted-lip-balm-with-hyaluronic-acid--shea-butter---clear/37205355/buy
52399,0.3% Retinol Face Serum - 30ml,ZERO THE START,https://www.myntra.com/serum-and-gel/zero+the+start/zero-the-start-03-retinol-face-serum-for-anti-ageing---30-ml/32756119/buy
52432,0.3% Retinol Face Serum - 30ml,INTIMIFY,https://www.myntra.com/serum-and-gel/intimify/intimify-03-retinol-face-serum-with-glutathione--hyaluronic-acid---30ml/25035140/buy
50508,1% Hyaluronic Sunscreen Gel,The Derma co.,https://www.myntra.com/face-sunscreen/the+derma+co./the-derma-co-1-hyaluronic-sunscreen-hydrating-gel---50g-in-vivo-tested/32772275/buy
50477,1% Hyaluronic Sunscreen Gel,The Derma co.,https://www.myntra.com/face-sunscreen/the+derma+co./the-derma-co-1-hyaluronic-sunscreen-oil-free-gel---50g-in-vivo-tested/32772276/buy
43153,1% Salicylic Acid Face Wash,LUVYH,https://www.myntra.com/face-wash-and-cleanser/luvyh/luvyh-glycolic--1-salicylic-acid-face-wash--100-ml/36791382/buy
40534,1% Salicylic Acid Face Wash,GLOWRITI,https://www.myntra.com/face-wash-and-cleanser/glowriti/glowriti-glycolic--1-salicylic-acid-face-wash---100-ml/42052091/buy
43384,1% Salicylic Acid Face Wash,LUVYH,https://www.myntra.com/face-wash-and-cleanser/luvyh/luvyh-glycolic--1-salicylic-acid-face-wash--100-ml/36749093/buy


#### Observation

Duplicate product names are expected in the Myntra dataset because product titles may not include sufficient information to distinguish different brands, shades, sizes, or variants.

Therefore, duplicate product names are not treated as duplicate records.

Product URLs are used as the primary identifier for distinguishing individual Myntra listings.

## 5. Logical Validation

After the cleaning steps are completed, the dataset is subjected to a set of logical and consistency checks.

These checks are used to identify potential issues in:

- **Pricing:** Checking whether discounted prices are logically consistent with original prices.
- **Ratings:** Checking whether rating values fall within the expected range.
- **Rating Counts:** Checking whether rating counts contain valid non-negative values.
- **Categorical Fields:** Reviewing platform, category, and subcategory values for consistency.

The validation results help confirm that the cleaned dataset is suitable for downstream analysis.

In [25]:
print(myntra_df.dtypes)

print(myntra_df["platform"].value_counts())

print(myntra_df["category"].value_counts())

print(myntra_df["sub_category"].value_counts())

print(
    "Invalid Prices:",
    (myntra_df["discounted_price"] >
     myntra_df["original_price"]).sum()
)

product_name            str
url                     str
category                str
sub_category            str
brand                   str
discounted_price    float64
original_price      float64
discount_percent    float64
rating_out_of_5     float64
rating_count          Int64
platform                str
dtype: object
platform
Myntra    80740
Name: count, dtype: int64
category
Lip Makeup          22220
Skin Care           18979
Hair Care           10201
Face Makeup          7992
Nail Makeup          6669
Body Care            5996
Eye Makeup           5907
Lip Care             2062
Hand & Foot Care      404
Eye Care              310
Name: count, dtype: int64
sub_category
Bullet Lipstick                9209
Mask & Peel                    6733
Nail Polish                    6669
Liquid Lipstick                6446
Face Wash & Cleanser           3952
Body Lotion                    3122
Serum & Gel                    3110
Foundation                     3080
Hair Oil                       

## 6. Export Cleaned Dataset

The cleaned and validated Myntra dataset is exported as a CSV file for use in subsequent analysis.

The exported dataset will serve as the Myntra input for the cross-platform analysis with the cleaned Nykaa dataset.

In [26]:
myntra_df.to_csv("myntra_products.csv", index=False)

## 7. Conclusion

The Myntra product dataset was cleaned and validated to prepare it for downstream analysis.

The cleaning process included converting numerical fields to appropriate data types, examining missing rating information, checking for duplicate records and URLs, and validating key product attributes.

The final dataset contains **80,740 product listings** with unique product URLs. Duplicate product names were retained because they can represent legitimate listings with different brands, shades, sizes, or variants.

The cleaned Myntra dataset is now ready to be combined with the previously cleaned Nykaa dataset for **cross-platform analysis**.